# Step 22.2 — RoPE：从二维旋转走到 Q/K 的相对位置

先在仓库根目录同步母版：

```bash
git pull
```

再用时间戳与 `cp -n` 建立独立工作副本；仓库已有 `notebooks/work/`，同名文件不覆盖：

```bash
cp -n notebooks/98_step22_2_rotary_position_embeddings.ipynb notebooks/work/98_step22_2_rotary_position_embeddings_$(date +%Y%m%d_%H%M%S)_work.ipynb
```

命令仅供未来使用，本次未执行。沿用 Python 3.12 / .venv，API 固定 **MLX v0.32.2**。

本节先讲二维旋转和点积，再把同一计算放进 d=4 的两个维度对，最后对应 reshape/broadcast/stack。原数学参考函数与观察单元原样保留，不替换 MiniGPT，不添加新的模型、缓存或执行入口。

2026-11-17 教学修订：**22.2 累计 1/2＝历史修正 0/2＋本次教学修订 1 次**。全部代码单元 execution_count=null、outputs=[]；没有执行课程代码、绘图、采样、测试、训练、实验或语料/checkpoint/work-copy 操作。纸算结果不代表运行通过。

## 1. 位置信息放在哪里？

[原 MiniGPT](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/7dfc7294da5eecc3bbae0836e98620c7cb5980e9/notebooks/65_step15_4_implement_minigpt.ipynb)把 token embedding 与 learned absolute position embedding 相加，再进入 Blocks。位置表第 p 行是可训练向量，表的行数限制了可索引的位置；[21.5](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/7dfc7294da5eecc3bbae0836e98620c7cb5980e9/notebooks/96_step21_5_context_cropping.ipynb)已解释窗口内重新编号的含义。

**RoPE（rotary position embedding，旋转位置编码）**在另一处引入位置：每个 head 已有 query/key 内容向量后，用位置决定的角度旋转 **Q 与 K**，然后再计算它们的点积。本文不旋转 V，也不在 token embedding 上再加一张位置表。

先记住这条位置关系，不急着写整个 attention：

```text
已有 Q/K (B,H,T,d) → 分别按各自位置旋转 → Q'/K' (B,H,T,d)
→ Q'K'^T / √d → 原有 mask / softmax → 加权原 V
```

本页 helper 只接收一个数组 x 并返回旋转后的数组，不知道 x 的角色是 Q 还是 K，不替调用者生成 Q/K/V，也不会自动应用 mask。学到的是一种位置变换，不能把这个独立函数当作完整 RoPE Transformer。

## 2. 二维旋转：先看一对坐标

把两个数看作平面向量 (a,b)。角度用 **radian（弧度）**：半圈是 π，四分之一圈是 π/2。单位圆上，cosθ 和 sinθ 是角度 θ 对应的横、纵坐标；例如 θ=0 时为 (1,0)，θ=π/2 时为 (0,1)。代码的 sin/cos 接收弧度，不接收“90 度”这个数值。

我们固定下面的旋转方向，把坐标当成列向量：

$$
R(\theta)=
\begin{pmatrix}\cos\theta&-\sin\theta\\
\sin\theta&\cos\theta\end{pmatrix},
\qquad
R(\theta)\begin{pmatrix}a\\b\end{pmatrix}
=\begin{pmatrix}a\cos\theta-b\sin\theta\\
a\sin\theta+b\cos\theta\end{pmatrix}.
$$

例如 (1,0) 旋转 π/2 得 (0,1)，(0,1) 旋转 π/2 得 (−1,0)。它改变方向，不放大向量：两项平方相加后，交叉项相消，留下 a²+b²，因为 cos²θ+sin²θ=1。

**Dot product（点积）**是对应坐标乘积之和：uᵀv=u₀v₀+u₁v₁。若两个向量都旋转同一个角度，R(θ)ᵀR(θ)=I，所以点积也不变；I 表示不改变向量的单位矩阵。但 Q 与 K 通常处于不同位置，接下来要处理的是两个不同角度。

## 3. 为什么是 q−p？先固定符号，再看手算

本节 **p 是 query 的位置，q 是 key 的位置**；Qₚ、Kq 是相应内容向量，不要把位置字母 q 当成向量 Q。先只看一个维度对，使用相同的角频率 ω（每移动一个位置增加多少弧度）：

$$
Q'_p=R(p\omega)Q_p,\qquad K'_q=R(q\omega)K_q.
$$

转置会把旋转变为反向旋转，两个旋转相乘会把角度相加，因此：

$$
(Q'_p)^\mathsf{T}K'_q
=Q_p^\mathsf{T}R(p\omega)^\mathsf{T}R(q\omega)K_q
=Q_p^\mathsf{T}R(-p\omega+q\omega)K_q
=Q_p^\mathsf{T}R((q-p)\omega)K_q.
$$

**q−p 的符号来自上面固定的矩阵和 QᵀK 顺序，不能随手反过来。** 用便于手算的角度检查：Q=[1,0] 的角度 α=0，K=[0,1] 的角度 β=π/2。旋转后 Q'=[1,0]、K'=[−1,0]，点积为 −1。按 β−α=π/2 计算 QᵀR(β−α)K 仍为 −1；误写为 α−β 则得到 +1。这里只人为选角度，**不是说默认 RoPE 的位置 1 对应 π/2**。

若保持内容向量不变，并给两个位置都加同一个位移，q−p 不变，以上点积也不变。这只说明旋转带来的**位置项**通过相对位移进入点积。Qₚ 和 Kq 的内容仍参与乘法；真实网络中它们还来自不同输入与前层表示，mask/softmax 也另有作用。不能把整个 attention 说成“只由距离决定”。

## 4. d=4：两个相邻维度对，两种角频率

令 d 为**每个 head**的维度，不是模型总宽度。d=4 时，本函数把 [x₀,x₁,x₂,x₃] 配成 (x₀,x₁) 与 (x₂,x₃)，pair index j 分别为 0、1。每一对都用第 2 节同一个二维旋转，但频率不同：

$$
\omega_j=\mathrm{base}^{-2j/d},
\qquad \theta_{t,j}=(\mathrm{offset}+t)\omega_j,
\qquad j=0,\ldots,d/2-1.
$$

base=10000、d=4 时，ω₀=1、ω₁=10000⁻¹ᐟ²=1/100。第一对每走一个位置转 1 弧度，第二对只转 0.01 弧度；这里不额外乘 2π。源码变量名 inv_freq 存的就是这些角频率系数。

**offset 是当前片段首项的绝对位置**，t 是片段内部从 0 开始的索引。比如 offset=1、T=3：

| 局部 t | 绝对位置 offset+t | 第 0 对角度 | 第 1 对角度 |
|---:|---:|---:|---:|
| 0 | 1 | 1 | 0.01 |
| 1 | 2 | 2 | 0.02 |
| 2 | 3 | 3 | 0.03 |

这张表说明普通片段的位置编号，不涉及 cache。offset 不是 token ID，也不是 head 编号；同一输入中的 batch/head 共享这个标量 offset。若未来拆片段，必须先声明各片段的起始位置，不能仅因局部索引又从 0 开始就把绝对位置也清零。

### 4.1 p≠q 的四维点积：把两对的贡献相加

从上表同一片段取 query 位置 **p=3**、key 位置 **q=1**。为便于手算，人为给定内容 Q₃=[1,0,0,1]、K₁=[0,1,1,0]；这些不是模型输出。较早 key 的位置 q 小于 p，但符号约定仍然是 q−p=−2。

| 维度对 | Q 对 / K 对 | query / key 角度 | 旋转后 Q 对 | 旋转后 K 对 | 该对点积 |
|---|---|---|---|---|---|
| j=0：维度 (0,1) | (1,0) / (0,1) | 3 / 1 | (cos3,sin3) | (−sin1,cos1) | −cos3 sin1+sin3 cos1=sin2 |
| j=1：维度 (2,3) | (0,1) / (1,0) | 0.03 / 0.01 | (−sin0.03,cos0.03) | (cos0.01,sin0.01) | −sin0.03 cos0.01+cos0.03 sin0.01=−sin0.02 |

所以完整点积是 **sin2−sin0.02**。也可从相对角度核对：第 0 对角差为 −2，QᵀR(−2)K=−sin(−2)=sin2；第 1 对角差为 −0.02，QᵀR(−0.02)K=sin(−0.02)。两种展开一致。

这是旋转后的未缩放点积。若进入 d=4 的 scaled dot-product attention，还要除以 √4=2，再处理可见性 mask 与 softmax；不是把这一个数直接当概率。sin 的自变量均为弧度，这些符号等式属于理想实数纸算，不是 float32 运行记录。

## 5. 公式怎样落到数组：先产生角度表

完整 helper 在第 6 节。这里按其原始源码顺序解释，不把摘录作为另一套独立函数。输入 x 的 shape 是 (B,H,T,d)；以 (1,1,3,4)、offset=1 为例：

```text
inv_freq = mx.exp(-math.log(base) * mx.arange(0, d, 2, dtype=mx.float32) / d)
positions = mx.arange(offset, offset + T, dtype=mx.float32)
angles = positions[:, None] * inv_freq[None, :]
c, s = mx.cos(angles), mx.sin(angles)
```

1. `arange(0,d,2)` 产生 0,2,…,d−2，即每对的 2j；exp(−log(base)·2j/d) 正好等于 base⁻²ʲᐟᵈ。因此 inv_freq 的 shape 是 (d/2,)=(2,)。
2. positions 是 (T,)=(3,)，表示上表的 [1,2,3]；None 插入长度为 1 的轴，不增加位置。
3. positions[:,None] 为 (3,1)，inv_freq[None,:] 为 (1,2)。**Broadcasting（广播）**让每个位置与每个频率相乘，得到 angles (3,2)，正是上表两列角度；这是逐元素乘法，不是点积。
4. c、s 与 angles 同形状。它们没有 batch/head 轴，因为同一组位置与频率在各 batch/head 间复用。

### 5.1 Reshape 只分组，不旋转、不重新配对

```text
pairs = x.astype(mx.float32).reshape(B, H, T, d // 2, 2)
a, b = pairs[..., 0], pairs[..., 1]
```

对一个位置的 [x₀,x₁,x₂,x₃]，reshape 成 [[x₀,x₁],[x₂,x₃]]。`d//2` 是整数除法，前面的保护检查已要求 d 为正偶数。末尾的 2 表示每个 pair 有两个坐标，不能把它误当成两个 head。

`...` 保留所有前面的轴；最后一轴取 0 得各对第一坐标 a=[x₀,x₂]，取 1 得第二坐标 b=[x₁,x₃]。所以 pairs 为 (B,H,T,d/2,2)，a 与 b 都是 (B,H,T,d/2)，具体为 (1,1,3,2)。

这两行的分组与取坐标操作不做旋转，只把相邻坐标安排成下一步能逐对运算的形状；c/s 已在上一段算好。astype 则把输入转换为 float32，是实际数值转换，不能与 reshape 的作用混为一谈。

### 5.2 广播做旋转，stack 放回两个坐标

```text
rotated = mx.stack([a * c - b * s, a * s + b * c], axis=-1)
return rotated.reshape(B, H, T, d)
```

a/b 的 (B,H,T,d/2) 与 c/s 的 (T,d/2) 从右侧对齐；c/s 相当于带有前导长度 1 的 batch/head 轴，因此每一对都拿到自己位置对应的 cos/sin。两个表达式分别就是第 2 节的 a cosθ−b sinθ 和 a sinθ+b cosθ。

它们各为 (B,H,T,d/2)。**stack** 在最后新增大小 2 的轴，把每一对的新第一/第二坐标放在一起，回到 (B,H,T,d/2,2)；最后 reshape 恢复 (B,H,T,d)，保持相邻交错顺序 [a'₀,b'₀,a'₁,b'₁]。

如果直接把两份 (…,d/2) concatenate 成 [a'₀,a'₁,b'₀,b'₁]，输出长度虽然也为 d，配对布局却变了。因此这里的 stack 不是可以随手替换的排版选择。整个过程不改变 B/H/T，也没有把所有 heads 混在一起旋转。

## 6. 原函数的保护与限制，再看完整代码

下面保留原短函数，不按行数拆造新 helper。签名中的 `*` 表示 offset/base 只能用关键字传入。

| 源码实际检查或操作 | 可以依赖的含义 |
|---|---|
| x.ndim==4，末轴为正偶数 | 专用 (B,H,T,d) 接口，完整 d 都旋转；不支持部分维度 |
| type(offset) is int 且 offset≥0 | 非负 Python 整数起点；不接受 bool 或每个 batch 一组 offset |
| math.isfinite(base) 且 base>0 | 正有限 base；没有强制 base>1，默认 10000 才是本文例子的频率 |
| x、positions、inv_freq 使用 float32 | 返回 float32，不承诺保持输入的混合精度 dtype |

讲解假设 x 是正常的有限实值 Q/K。**函数没有检查 x 的数值有限性，也没有专门检查转换或旋转后的数值**，不能把“shape/base/offset 检查通过”当成数值健康证明。极大位置在 float32 中可能不能区分相邻整数；极端 base/角度也会带来精度或非有限值问题。理想旋转保持范数，不等于这份浮点实现逐位精确保持范数。

本函数没有 scale 参数、可替换频率表或部分 rotary dimensions，也没有 attention、mask、cache 与模型状态管理。本轮不添加这些功能。

In [ ]:
import math
import mlx.core as mx

# 本单元及后续所有单元均为未执行的教学源码。
# 仅在未来独立工作副本中按顺序运行；本次材料制作没有导入 MLX。

def rope_adjacent(x, *, offset=0, base=10000.0):
    if x.ndim != 4 or x.shape[-1] % 2 or x.shape[-1] == 0:
        raise ValueError("expect (B,H,T,d) with positive even d")
    if type(offset) is not int or offset < 0:
        raise ValueError("offset must be a nonnegative integer")
    if not math.isfinite(base) or base <= 0:
        raise ValueError("base must be positive and finite")
    B, H, T, d = x.shape
    inv_freq = mx.exp(-math.log(base) * mx.arange(0, d, 2, dtype=mx.float32) / d)
    positions = mx.arange(offset, offset + T, dtype=mx.float32)
    angles = positions[:, None] * inv_freq[None, :]
    c, s = mx.cos(angles), mx.sin(angles)
    pairs = x.astype(mx.float32).reshape(B, H, T, d // 2, 2)
    a, b = pairs[..., 0], pairs[..., 1]
    rotated = mx.stack([a * c - b * s, a * s + b * c], axis=-1)
    return rotated.reshape(B, H, T, d)

## 7. 可选未来观察：它能说明什么，又说明不了什么？

保留原观察单元：x 的 shape 为 (1,1,2,4)，两个位置的内容相同，offset=0、base 使用默认值。未来先用纸笔判断：第 0 个位置角度为 0，是否应保持原向量？第 1 个位置两个 pair 为什么转得不同，但理想实数下总平方范数仍不变？

`mx.sum(x*x,axis=-1)` 只沿最后的特征轴求平方和，before_norm2/after_norm2 都为 (1,1,2)。未来浮点比较应考虑误差，不能要求打印十进制或逐位完全一样。本次没有运行、绘图或填写输出。

这个单元只观察一个 x 的形状和范数，**没有构造 Q/K 两个数组，也没有计算 p≠q 的点积**；不能把它当成已经验证第 4.1 节的相对位置公式，更不能据此证明模型质量、长序列能力或 fused 实现等价。第 4.1 节仍是独立的纸算。

In [ ]:
x = mx.array([[[[1., 0., 0., 1.], [1., 0., 0., 1.]]]])
y = rope_adjacent(x, offset=0)
before_norm2 = mx.sum(x * x, axis=-1)
after_norm2 = mx.sum(y * y, axis=-1)
mx.eval(y, before_norm2, after_norm2)
print("shape:", y.shape)
print("norm² before/after:", before_norm2.tolist(), after_norm2.tolist())

## 8. 对照固定 MLX：配对方式必须说清楚

固定 [MLX v0.32.2 的绑定](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/fast.cpp)及[计算源码](https://github.com/ml-explore/mlx/blob/v0.32.2/mlx/fast.cpp)显示：

| 配对约定 | d=4 时的坐标对 | 与本 helper 的关系 |
|---|---|---|
| traditional=True | (0,1)、(2,3) | 相邻配对，与本节一致 |
| traditional=False | (0,2)、(1,3) | 分半配对，不能拿相同坐标布局直接冒充本函数 |

仅为说明对应参数，fast 接口的参考写法如下；本页没有执行或新增这一调用：

```text
mx.fast.rope(x, d, traditional=True, base=10000.0, scale=1.0, offset=offset)
```

在这个底层接口里，traditional、base、scale、offset 都要显式提供为关键字；这里 dims=d、scale=1，与本 helper 的全维度/未缩放位置约定对应。fast.rope 还支持本 helper 没有的参数组合，不在本节实现。

[nn.RoPE 的包装层](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/nn/layers/positional_encoding.py)却有默认值：traditional=False。因此 `nn.RoPE(d)` 不会自动采用本节的相邻配对。比较时还需对齐 dtype、维度、base、位置和 scale；本 helper 固定转 float32，而 fast 路径保持输入浮点 dtype。即便数学约定一致，也不承诺逐位相同或已经测得加速。

## 9. 这是架构选择，不是旧 checkpoint 的开关

当前已验收的 MiniGPT 仍使用输入端 learned position embedding。本节没有删除它、旋转真实模型的 Q/K、读取旧权重或修改 Adam 状态。不能把位置表删掉再加载原 checkpoint，就宣称得到了等价的 RoPE 模型；也不能悄悄把旧位置表与新旋转同时混入 baseline，然后把结果归因于本节公式。

换位置机制需要明确架构、训练或有依据的权重转换方案。公式能为更大绝对位置计算角度，不等于模型已经学到任意长序列；训练覆盖、内容分布和数值精度仍有限制，本次没有相关评测。

offset 在本页只表示片段首项的位置起点，不意味着已经有 KV cache。如果以后研究缓存，位置编号与可见性还需另行约定；[21.5 的重新计算窗口](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/7dfc7294da5eecc3bbae0836e98620c7cb5980e9/notebooks/96_step21_5_context_cropping.ipynb)也不会因加一个 offset 就自动变成等价的缓存算法。本节不要求实现缓存工程。

## 10. 自己检查理解（不附答案）

1. 手算 (2,1) 旋转 π/2 后的坐标与平方范数，说明哪一步体现了旋转方向。
2. d=4、base=10000，取 p=2、q=0、Q₂=[1,1,0,0]、K₀=[1,0,1,0]，分别写出两对的角度与点积贡献；明确采用 q−p 的依据，不需要小数近似。
3. 一个普通片段的 x.shape=(2,3,4,4)、offset=5。写出 positions 的数值，以及 angles、a/b、stack 后数组的 shape；这不是缓存问题，不要求实现代码。

先写自己的推理，练习不附答案。未来观察、纸算与代码静态审阅各能支持不同结论，不把“没有报错”代替理解。

## 11. 依据与材料状态

- [RoFormer 原论文](https://arxiv.org/abs/2104.09864)：旋转位置编码的来源。本节的坐标例子与逐步展开是教学用纸算。
- [MLX v0.32.2 fast.cpp 绑定](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/fast.cpp)：必需关键字、dims、offset 与接口说明。
- [MLX v0.32.2 mlx/fast.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/mlx/fast.cpp)：相邻/分半配对、旋转方向及 dtype 行为。
- [MLX v0.32.2 positional_encoding.py](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/nn/layers/positional_encoding.py)：nn.RoPE 的默认配对与参数传递。
- [MLX v0.32.2 ops.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/ops.cpp)：arange、reshape、stack 与逐元素运算。

本次只修订讲解与本节范围内的练习；两个原代码单元逐字保留，没有新增可执行入口。严格 JSON/schema、合法唯一 ID、静态语法/AST 与变更范围检查属于静态审核，不是执行验证。

22.2 累计 **1/2**（历史 0/2＋本次教学修订 1 次），其他历史计数不变。所有代码单元未执行、输出为空；没有运行 MLX、绘图、测试、训练、实验、语料读取/下载或 checkpoint/work-copy 操作。本轮停在 22.2，等待独立验收。